# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Козак Наталя'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-34'
НОМЕР        = 6       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 6   ·   Козак Наталя   ·   ШІ-34

Ваші вісім стовпців:
   1. ПІП
   2. Примітка
   3. Перебіг
   4. Дата розтину
   5. Ліжко днів
   6. Т вище38
   7. МНВ1
   8. Креат2

Ваша пара для аналізу зв’язку: Ліжко днів та Вік

Ваше додаткове питання:
   Скільки стовпців файлу є ідентифікаторами? Як їх розпізнати автоматично, не дивлячись на назви?


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,ПІП,Примітка,Перебіг,Дата розтину,Ліжко днів,Т вище38,МНВ1,Креат2
0,Федорюк Петро Юрійович,NaN,важкий,2022-03-11,14,4 дні до госп,3.10,92
1,Загул Григорій Сергійович,"отримував актемра, часті носові кровотечі, втрата нюху",важкий,NaT,22,5 до гос,1.22,73
2,Бондара Лариса Павлівна,часта втрата свідомосі в стаціонарі,важкий,NaT,20,10 до гос,1.03,117.2
3,Ватаманюка Уляна Дмитрівна,панкреонекроз,важкий,NaT,20,9 до гос,1.69,105.1
4,Бабійа Лариса Олегівна,NaN,важкий,NaT,27,NaN,1.15,101
5,Ковалика ганна Андріївна,NaN,важкий,2021-12-26,9(5 у реан),NaN,0.90,172
6,Височан Віктор Володимирович,"глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс",важкий,NaT,18,7 дні до госп,1.09,97.8
7,Дзюбаа Ірина Іванівна,NaN,серед важ,NaT,10,9 днів до госп,0.94,107


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   ПІП           121 non-null    str           
 1   Примітка      45 non-null     str           
 2   Перебіг       121 non-null    str           
 3   Дата розтину  61 non-null     datetime64[us]
 4   Ліжко днів    120 non-null    object        
 5   Т вище38      77 non-null     object        
 6   МНВ1          90 non-null     float64       
 7   Креат2        102 non-null    object        
dtypes: datetime64[us](1), float64(1), object(3), str(3)
memory usage: 7.7+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- ПІП
    непорожніх 121 з 121 | унікальних 121 | типи: str×121
    приклади: ["'Федорюк Петро Юрійович'", "'Загул Григорій Сергійович'", "'Бондара Лариса Павлівна'", "'Ватаманюка Уляна Дмитрівна'"]

--- Примітка
    непорожніх 45 з 121 | унікальних 44 | типи: str×45
    приклади: ["'отримував актемра, часті носові кровотечі, втрата нюху'", "'часта втрата свідомосі в стаціонарі'", "'панкреонекроз'", "'глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс'"]

--- Перебіг
    непорожніх 121 з 121 | унікальних 3 | типи: str×121
    приклади: ["'важкий'", "'серед важ'", "'вкрай важкий '"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- Ліжко днів
    непорожніх 120 з 121 | унікальних 40 | типи: int×119, str×1
    приклади: ['14', '22', '20', '27']

--- Т вище38
    непорожніх 77 з 121 | унікаль

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    'ПІП': ('ідентифікатор',
            'Прізвище, ім\'я та по батькові лише вказують на конкретного пацієнта, усі 121 значення різні, жодної величини за ними немає.'),
    'Примітка': ('номінальна',
                 'Вільний текст про особливості пацієнта (алергія, супутні стани), записи можна лише порівняти на збіг, порядку між ними немає.'),
    'Перебіг': ('порядкова',
                'Тяжкість перебігу має природний порядок "серед важ" < "важкий" < "вкрай важкий", але відстань між сусідніми рівнями невідома.'),
    'Дата розтину': ('інтервальна',
                     'Різниця між датами має сенс (у днях), а нуль календаря умовний, тому "удвічі пізніша дата" не має змісту.'),
    'Ліжко днів': ('відношень',
                   'Тривалість перебування в стаціонарі має природний нуль (0 днів), і 20 днів справді вдвічі довше за 10.'),
    'Т вище38': ('абсолютна',
                 'Це лічба днів з температурою вище 38 градусів: і нуль (жодного дня), і одиниця (один день) природні, значення не залежать від вибору одиниць.'),
    'МНВ1': ('відношень',
             'Міжнародне нормалізоване відношення - безрозмірне відношення протромбінового часу до норми, має природний нуль і відношення мають сенс.'),
    'Креат2': ('відношень',
               'Концентрація креатиніну в мкмоль/л має природний нуль (речовини немає), тому вдвічі більша концентрація має фізичний зміст.'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,ПІП,ідентифікатор,"Прізвище, ім'я та по батькові лише вказують на конкретно..."
1,Примітка,номінальна,"Вільний текст про особливості пацієнта (алергія, супутні..."
2,Перебіг,порядкова,"Тяжкість перебігу має природний порядок ""серед важ"" < ""в..."
3,Дата розтину,інтервальна,"Різниця між датами має сенс (у днях), а нуль календаря у..."
4,Ліжко днів,відношень,Тривалість перебування в стаціонарі має природний нуль (...
5,Т вище38,абсолютна,Це лічба днів з температурою вище 38 градусів: і нуль (ж...
6,МНВ1,відношень,Міжнародне нормалізоване відношення - безрозмірне віднош...
7,Креат2,відношень,Концентрація креатиніну в мкмоль/л має природний нуль (р...


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
# Порядок рівнів для порядкової шкали задаю вручну (алфавіт тут не підходить)
ПОРЯДОК = {'Перебіг': ['серед важ', 'важкий', 'вкрай важкий']}


def до_числа(s):
    """Дістає перше число із запису: '9(5 у реан)' -> 9, '4 дні до госп' -> 4, 'немає' -> NaN."""
    return pd.to_numeric(s.astype(str).str.extract(r'(\d+(?:[.,]\d+)?)')[0].str.replace(',', '.'),
                         errors='coerce')


def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р

    # мода й частоти - для всіх шкал
    if ш == 'порядкова':
        з = з.astype(str).str.strip()
    elif ш in ('відношень', 'абсолютна'):
        з = до_числа(з).dropna()
    р['мода'] = з.mode().iloc[0]
    р['частоти (топ-3)'] = з.value_counts().head(3).to_dict()

    if ш == 'номінальна':
        return р

    # порядкова: медіана та квартилі через коди із заданим порядком
    if ш == 'порядкова':
        рівні = ПОРЯДОК[стовпець]
        коди = з.map({v: i for i, v in enumerate(рівні)})
        р['медіана'] = рівні[int(коди.quantile(0.5, interpolation='lower'))]
        р['Q1'] = рівні[int(коди.quantile(0.25, interpolation='lower'))]
        р['Q3'] = рівні[int(коди.quantile(0.75, interpolation='lower'))]
        return р

    р['медіана'] = з.median()
    р['Q1'], р['Q3'] = з.quantile(0.25), з.quantile(0.75)
    р['середнє'] = з.mean()
    р['стд. відхилення'] = з.std()

    if ш == 'інтервальна':
        return р

    # відношень і абсолютна
    р['коеф. варіації, %'] = round(100 * з.std() / з.mean(), 1)
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- ПІП  (ідентифікатор)
    {'непорожніх': 121, 'унікальних': 121}

--- Примітка  (номінальна)
    {'непорожніх': 45, 'унікальних': 44, 'мода': 'Поліалергетик', 'частоти (топ-3)': {'Поліалергетик': 2, 'отримував актемра, часті носові кровотечі, втрата нюху': 1, 'часта втрата свідомосі в стаціонарі': 1}}

--- Перебіг  (порядкова)
    {'непорожніх': 121, 'унікальних': 3, 'мода': 'важкий', 'частоти (топ-3)': {'важкий': 90, 'серед важ': 30, 'вкрай важкий': 1}, 'медіана': 'важкий', 'Q1': 'важкий', 'Q3': 'важкий'}

--- Дата розтину  (інтервальна)
    {'непорожніх': 61, 'унікальних': 50, 'мода': Timestamp('2022-06-07 00:00:00'), 'частоти (топ-3)': {Timestamp('2022-06-07 00:00:00'): 3, Timestamp('2022-04-07 00:00:00'): 2, Timestamp('2022-04-27 00:00:00'): 2}, 'медіана': Timestamp('2022-04-23 00:00:00'), 'Q1': Timestamp('2022-03-29 00:00:00'), 'Q3': Timestamp('2022-05-07 00:00:00'), 'середнє': Timestamp('2022-04-06 13:22:37.377049'), 'стд. відхилення': Timedelta('72 days 13:34:38.466302')}

--

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [9]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
ПІП,121,121,Федорюк Петро Юрійович,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Примітка,45,44,Поліалергетик,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Перебіг,121,3,важкий,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
Ліжко днів,120.0,40.0,10.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Т вище38,77.0,28.0,3.0,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
МНВ1,90.0,NaN,NaN,NaN,1.333778,0.8,1.0525,1.19,1.34,10.6,1.032131
Креат2,102.0,72.0,77.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# 1) мода ідентифікатора
мода_піп = df['ПІП'].describe()['top']
# 2) кількість "унікальних" днів температури, коли одне число записано різними способами
унік_т = df['Т вище38'].nunique()
реально_різних = до_числа(df['Т вище38']).nunique()
# 3) коефіцієнт варіації дати (дні від 01.01.1970)
дні = (df['Дата розтину'].dropna() - pd.Timestamp('1970-01-01')).dt.days
cv_дати = round(100 * дні.std() / дні.mean(), 3)

БЕЗГЛУЗДІ = [
    ('ПІП', 'мода (top у describe)', мода_піп,
     'Усі 121 ПІП різні (freq = 1), тому "найчастіше значення" - це просто перший рядок таблиці, а не типовий пацієнт.'),
    ('Т вище38', 'кількість унікальних (unique)', унік_т,
     f'Одне й те саме число днів записано по-різному (3, "3 дні до госп", "3дні вдома"), реально різних значень лише {реально_різних}, а не {унік_т}.'),
    ('Дата розтину', 'коефіцієнт варіації, %', cv_дати,
     'Шкала інтервальна, нуль (1970 рік) умовний: з іншим початком відліку CV буде будь-яким, тому він нічого не каже про розкид дат.'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,ПІП,мода (top у describe),Федорюк Петро Юрійович,"Усі 121 ПІП різні (freq = 1), тому ""найчастіше значення""..."
1,Т вище38,кількість унікальних (unique),28,"Одне й те саме число днів записано по-різному (3, ""3 дні..."
2,Дата розтину,"коефіцієнт варіації, %",0.38,"Шкала інтервальна, нуль (1970 рік) умовний: з іншим поча..."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Ліжко днів × Вік

Ліжко днів: шкала — відношень
Вік: шкала — не у вашому списку

    Ліжко днів  Вік
0           14   55
1           22   43
2           20   54
3           20   68
4           27   65
5  9(5 у реан)   66


In [12]:
from scipy import stats

ОБРАНА_МІРА = 'кореляція Пірсона'
ЧОМУ_САМЕ_ВОНА = ('Обидві змінні мають шкалу відношень (дні та роки з природним нулем), '
                  'тому для них допустимі середнє та відхилення, а отже й коефіцієнт Пірсона.')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

ліжко = до_числа(df[A])      # '9(5 у реан)' -> 9
вік = df[B]
м = ліжко.notna() & вік.notna()

r, p = stats.pearsonr(ліжко[м], вік[м])
print(f'n = {м.sum()}, r Пірсона = {r:.3f}, p = {p:.3f}')

# перевірка: розподіл ліжко-днів сильно скошений (медіана 15, максимум 79),
# тому дивлюсь ще й на ранговий коефіцієнт, менш чутливий до викидів
rs, ps = stats.spearmanr(ліжко[м], вік[м])
print(f'для порівняння r Спірмена = {rs:.3f}, p = {ps:.3f}')

n = 120, r Пірсона = -0.100, p = 0.279
для порівняння r Спірмена = -0.194, p = 0.034


In [13]:
# Пірсон для порядкової змінної "Перебіг" та "Ліжко днів".
# Беру три кодування рівнів, кожне з них виглядає "законно".
кодування = {
    'за порядком 1-2-3':     {'серед важ': 1, 'важкий': 2, 'вкрай важкий': 3},
    'за порядком 1-2-10':    {'серед важ': 1, 'важкий': 2, 'вкрай важкий': 10},
    'за алфавітом (pandas)': {'важкий': 0, 'вкрай важкий': 1, 'серед важ': 2},
}
перебіг = df['Перебіг'].str.strip()
for назва, к in кодування.items():
    print(f'{назва:24s} r = {перебіг.map(к)[м].corr(ліжко[м]):.3f}')

# Ті самі дані, а r змінюється і навіть міняє знак: Пірсон використовує відстані між кодами,
# а для порядкової шкали відстаней не існує. Тому таке число не можна інтерпретувати.

за порядком 1-2-3        r = 0.182
за порядком 1-2-10       r = 0.063
за алфавітом (pandas)    r = -0.202


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [14]:
ПІДГОТОВКА = {
    'ПІП': 'не беремо в модель, бо це ідентифікатор пацієнта',
    'Примітка': 'вільний текст: у модель не беремо, за потреби - бінарна ознака "є примітка" або one-hot ключових слів',
    'Перебіг': 'цілі коди із заданим вручну порядком: серед важ=0, важкий=1, вкрай важкий=2',
    'Дата розтину': 'для прогнозу летальності не беремо (є лише у померлих, витік цілі); в інших задачах - тривалість від поступлення в днях',
    'Ліжко днів': 'витягти число з тексту, логарифмувати (скошений розподіл) і стандартизувати',
    'Т вище38': 'витягти число днів з тексту, нормалізувати min-max до [0, 1]',
    'МНВ1': 'перевірити викид 10.6, потім стандартизувати',
    'Креат2': '"немає" замінити на NaN, логарифмувати й стандартизувати',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,ПІП,ідентифікатор,"не беремо в модель, бо це ідентифікатор пацієнта"
1,Примітка,номінальна,"вільний текст: у модель не беремо, за потреби - бінарна ..."
2,Перебіг,порядкова,"цілі коди із заданим вручну порядком: серед важ=0, важки..."
3,Дата розтину,інтервальна,"для прогнозу летальності не беремо (є лише у померлих, в..."
4,Ліжко днів,відношень,"витягти число з тексту, логарифмувати (скошений розподіл..."
5,Т вище38,абсолютна,"витягти число днів з тексту, нормалізувати min-max до [0..."
6,МНВ1,відношень,"перевірити викид 10.6, потім стандартизувати"
7,Креат2,відношень,"""немає"" замінити на NaN, логарифмувати й стандартизувати"


In [15]:
підготовлені = pd.DataFrame(index=df.index)

# 1) порядкова: "Перебіг" -> коди із заданим порядком
підготовлені['Перебіг_код'] = (df['Перебіг'].str.strip()
                               .map({v: i for i, v in enumerate(ПОРЯДОК['Перебіг'])}))

# 2) відношень: "Ліжко днів" -> log, потім стандартизація
лог = np.log1p(до_числа(df['Ліжко днів']))
підготовлені['Ліжко_днів_z'] = (лог - лог.mean()) / лог.std()

# 3) абсолютна: "Т вище38" -> число днів, нормалізація min-max (перед цим перевіряю варіативність)
т = до_числа(df['Т вище38'])
assert т.max() > т.min(), 'сталий стовпець, нормалізувати не можна'
підготовлені['Т_вище38_мінмакс'] = (т - т.min()) / (т.max() - т.min())

підготовлені.describe().round(3)

,Перебіг_код,Ліжко_днів_z,Т_вище38_мінмакс
count,121.000,120.000,77.000
mean,0.760,-0.000,0.358
std,0.448,1.000,0.206
min,0.000,-3.143,0.000
25%,1.000,-0.522,0.222
50%,1.000,0.054,0.333
75%,1.000,0.611,0.444
max,2.000,2.528,1.000


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [17]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [18]:
ВИПРАВЛЕННЯ = [
    (1, 'Середнє від номінальної змінної "помер/жив" (коди 1 і 2)',
     '1.50 залежить лише від вибору кодів: з кодами 0/1 вийшло б 0.50, а "посередині між смертю і життям" пацієнт бути не може',
     'Рахувати частоти: частка померлих 61 з 121 = 50.4%'),
    (2, 'Дані не придатні: одна стать записана різними способами (ж/жін, ч/чол)',
     '4 - це кількість варіантів написання, а не кількість статей, частки груп теж неправильні',
     'Уніфікувати записи (перша літера) і тоді рахувати частоти: ж 75, ч 46'),
    (3, 'Порядкова шкала відсортована за алфавітом, з дублями через пробіли та одруківки, і від неї взято середнє',
     '"Найлегший - важкий, найважчий - середній" і середнє 5.9 залежать від алфавіту та кодів, а не від тяжкості',
     'Очистити написання, задати порядок легкий < середній < важкий < вкрай важкий < критичний, рахувати моду й медіану'),
    (4, 'Дані не придатні: 17 записів КТ є текстом-діапазоном ("70-80%"), to_numeric мовчки їх викинув',
     'Середнє 39.6% пораховане без 14% пацієнтів, причому саме тих, у кого ураження описане діапазоном',
     'Перетворити діапазони на середину інтервалу, а через скошений розподіл повідомляти медіану'),
    (5, 'Дані не придатні: змішано одиниці, значення 0.9 записане як частка, а не 90%',
     'Мінімум 0.9 неможливий для живої людини, він тягне середнє вниз, тому 82.05% не описує "більшість" пацієнтів',
     'Значення <= 1 помножити на 100, потім рахувати медіану (вона й описує "більшість")'),
    (6, 'Коефіцієнт варіації для дат, а дата - інтервальна шкала з умовним нулем (1970 рік)',
     '0.595% показує лише те, що дати далеко від 1970 року; з іншим початком календаря вийшло б зовсім інше число',
     'Розкид дат описувати стандартним відхиленням у днях, розмахом або розподілом за місяцями'),
    (7, 'Кореляція Пірсона між двома номінальними змінними (стать і летальність)',
     'Знак r залежить від того, як закодовано стать, а "лінійний зв\'язок" для категорій не має змісту; висновок "близько до нуля" не перевірено тестом',
     'Таблиця спряженості, критерій хі-квадрат і коефіцієнт Крамера V'),
    (8, 'Min-max нормалізація всіх числових стовпців без перевірки, зокрема сталого ФЕРИТ2 і номінального "помер/жив"',
     'Для ФЕРИТ2 max = min, знаменник 0, і весь стовпець мовчки став NaN; нормалізований код результату не має змісту',
     'Перед нормалізацією перевірити варіативність, сталі стовпці й номінальні коди виключити'),
    (9, 'Припущення, що в наборі є повністю заповнені рядки, хибне: багато стовпців майже порожні',
     '0 рядків - це не "чистий набір", а повна втрата даних, модель будувати нема на чому',
     'Спочатку прибрати стовпці з великою часткою пропусків, решту пропусків заповнити або видалити за окремими стовпцями'),
    (10, 'Дата розтину є лише у померлих (витік цілі), а NaT перетворено на int64 як величезне від\'ємне число',
     'r = 1.00 означає лише "є дата розтину = помер", це наслідок результату, а не його предиктор',
     'Виключити "Дату розтину" з ознак і рахувати зв\'язок лише для ознак, відомих до результату'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,"Середнє від номінальної змінної ""помер/жив"" (коди 1 і 2)",1.50 залежить лише від вибору кодів: з кодами 0/1 вийшло...,Рахувати частоти: частка померлих 61 з 121 = 50.4%
1,2,Дані не придатні: одна стать записана різними способами ...,"4 - це кількість варіантів написання, а не кількість ста...",Уніфікувати записи (перша літера) і тоді рахувати частот...
2,3,"Порядкова шкала відсортована за алфавітом, з дублями чер...","""Найлегший - важкий, найважчий - середній"" і середнє 5.9...","Очистити написання, задати порядок легкий < середній < в..."
3,4,"Дані не придатні: 17 записів КТ є текстом-діапазоном (""7...","Середнє 39.6% пораховане без 14% пацієнтів, причому саме...","Перетворити діапазони на середину інтервалу, а через ско..."
4,5,"Дані не придатні: змішано одиниці, значення 0.9 записане...","Мінімум 0.9 неможливий для живої людини, він тягне серед...","Значення <= 1 помножити на 100, потім рахувати медіану (..."
5,6,"Коефіцієнт варіації для дат, а дата - інтервальна шкала ...","0.595% показує лише те, що дати далеко від 1970 року; з ...","Розкид дат описувати стандартним відхиленням у днях, роз..."
6,7,Кореляція Пірсона між двома номінальними змінними (стать...,"Знак r залежить від того, як закодовано стать, а ""лінійн...","Таблиця спряженості, критерій хі-квадрат і коефіцієнт Кр..."
7,8,Min-max нормалізація всіх числових стовпців без перевірк...,"Для ФЕРИТ2 max = min, знаменник 0, і весь стовпець мовчк...","Перед нормалізацією перевірити варіативність, сталі стов..."
8,9,"Припущення, що в наборі є повністю заповнені рядки, хибн...","0 рядків - це не ""чистий набір"", а повна втрата даних, м...","Спочатку прибрати стовпці з великою часткою пропусків, р..."
9,10,"Дата розтину є лише у померлих (витік цілі), а NaT перет...","r = 1.00 означає лише ""є дата розтину = помер"", це наслі...","Виключити ""Дату розтину"" з ознак і рахувати зв'язок лише..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [19]:
# Розділ 1: середнє коду vs частка померлих
print('Розділ 1')
print('  неправильно: середнє коду =', round(df['помер/жив'].mean(), 2))
print('  правильно:   частка померлих =', round(100 * (df['помер/жив'] == 1).mean(), 1), '%')
print()

# Розділ 2: 4 "статі" vs 2 після уніфікації
print('Розділ 2')
print('  неправильно:', df['Стать'].nunique(), 'статі')
стать = df['Стать'].astype(str).str.strip().str[0].map({'ж': 'жіноча', 'ч': 'чоловіча'})
print('  правильно:  ', стать.value_counts().to_dict())
print()

# Розділ 7: Пірсон vs хі-квадрат і V Крамера
print('Розділ 7')
помер = (df['помер/жив'] == 1).astype(int)
print('  неправильно: r Пірсона =', round(стать.map({'чоловіча': 0, 'жіноча': 1}).corr(помер), 3),
      ', а з кодами навпаки:', round(стать.map({'чоловіча': 1, 'жіноча': 0}).corr(помер), 3))
таблиця = pd.crosstab(стать, df['помер/жив'].map({1: 'помер', 2: 'вижив'}))
хі2, p, _, _ = stats.chi2_contingency(таблиця, correction=False)
V = np.sqrt(хі2 / (таблиця.values.sum() * (min(таблиця.shape) - 1)))
print(таблиця.to_string())
print(f'  правильно:   хі-квадрат = {хі2:.2f}, p = {p:.3f}, V Крамера = {V:.3f}')

Розділ 1
  неправильно: середнє коду = 1.5
  правильно:   частка померлих = 50.4 %

Розділ 2
  неправильно: 4 статі
  правильно:   {'жіноча': 75, 'чоловіча': 46}

Розділ 7
  неправильно: r Пірсона = 0.177 , а з кодами навпаки: -0.177
помер/жив  вижив  помер
Стать                  
жіноча        32     43
чоловіча      28     18
  правильно:   хі-квадрат = 3.78, p = 0.052, V Крамера = 0.177


---
# Етап 7. Висновки та індивідуальне питання

In [20]:
ВИСНОВКИ = [
    'Тривалість госпіталізації ("Ліжко днів") майже не пов\'язана з віком: r Пірсона = -0.10 при p = 0.28, тобто лінійного зв\'язку не виявлено, хоча ранговий коефіцієнт показує слабку від\'ємну тенденцію.',
    'Стовпці "Ліжко днів", "Т вище38" і "Креат2" мають шкалу відношень або абсолютну, але pandas читає їх як object через текстові записи ("9(5 у реан)", "4 дні до госп", "немає"), тому без очищення їхні статистики не рахуються.',
    'Для "Перебігу" допустимі лише мода й медіана (обидві - "важкий"), а Пірсон з різними кодуваннями рівнів дає r від -0.20 до 0.18, тобто для порядкової шкали це число нічого не означає.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Тривалість госпіталізації ("Ліжко днів") майже не пов'язана з віком: r Пірсона = -0.10 при p = 0.28, тобто лінійного зв'язку не виявлено, хоча ранговий коефіцієнт показує слабку від'ємну тенденцію.
2. Стовпці "Ліжко днів", "Т вище38" і "Креат2" мають шкалу відношень або абсолютну, але pandas читає їх як object через текстові записи ("9(5 у реан)", "4 дні до госп", "немає"), тому без очищення їхні статистики не рахуються.
3. Для "Перебігу" допустимі лише мода й медіана (обидві - "важкий"), а Пірсон з різними кодуваннями рівнів дає r від -0.20 до 0.18, тобто для порядкової шкали це число нічого не означає.


In [21]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = (
    'У файлі три ідентифікатори: "номер і/х" (номер історії хвороби), "ПІП" і "Телефон". '
    'Вони не є величинами, а лише вказують на конкретного пацієнта. '
    'Автоматично їх можна розпізнати без назв за такими ознаками: '
    '1) частка унікальних значень близька до 100% (кожен рядок має своє значення); '
    '2) стовпець майже повністю заповнений, на відміну від рідкісних аналізів, де унікальність висока лише через пропуски; '
    '3) значення є довгими рядками або "числами" з нецифровими символами (дробова риска, пробіли, слова), '
    'тому арифметика над ними неможлива. '
    'Дійсні числові стовпці (наприклад вік) теж можуть мати багато унікальних значень, але в них є повтори, '
    'тому їх відсіює поріг унікальності. Поріг беру 95%, а не 100%, бо в "номері і/х" є два повтори через помилки введення. '
    'Перевірка в коді нижче.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)
print()

частка_унік = df.nunique() / df.notna().sum()
заповненість = df.notna().mean()
кандидати = частка_унік[(частка_унік >= 0.95) & (заповненість >= 0.9)]
print('Автоматично знайдені ідентифікатори:', list(кандидати.index))

ВАШЕ ПИТАННЯ:
Скільки стовпців файлу є ідентифікаторами? Як їх розпізнати автоматично, не дивлячись на назви?

У файлі три ідентифікатори: "номер і/х" (номер історії хвороби), "ПІП" і "Телефон". Вони не є величинами, а лише вказують на конкретного пацієнта. Автоматично їх можна розпізнати без назв за такими ознаками: 1) частка унікальних значень близька до 100% (кожен рядок має своє значення); 2) стовпець майже повністю заповнений, на відміну від рідкісних аналізів, де унікальність висока лише через пропуски; 3) значення є довгими рядками або "числами" з нецифровими символами (дробова риска, пробіли, слова), тому арифметика над ними неможлива. Дійсні числові стовпці (наприклад вік) теж можуть мати багато унікальних значень, але в них є повтори, тому їх відсіює поріг унікальності. Поріг беру 95%, а не 100%, бо в "номері і/х" є два повтори через помилки введення. Перевірка в коді нижче.

Автоматично знайдені ідентифікатори: ['номер і/х', 'ПІП', 'Телефон']


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [22]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'текст, як є',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'текст, як є',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'текст, як є',
    'етап 7 — висновки й відповідь':   'текст, як є',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = ('Код і тексти згенерував Claude. Я перевірила, що ноутбук виконується згори донизу на чистому ядрі, '
                      'звірила числа у висновках з виходами клітинок, переконалась, що до_числа правильно витягує '
                      'числа з текстових записів, і що порядок рівнів "Перебігу" заданий вручну, а не за алфавітом.')
ПІДТВЕРДЖУЮ = True      # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [23]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 6 готова до захисту.
Козак Наталя, ШІ-34


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.